# Using foambryo

In this notebook, we'll see:
- How to install foambryo,
- What are the inputs of foambryo,
- How to compute relative tensions and pressures inside cells,
- How to plot these forces using the foambryo viewer.

# 0) Installation

Foambryo is available on pyPI such that you can just use pip !

- if you don't need any visualization, use `pip install foambryo`.
- if you want to use the foambryo viewer to visualize the forces on a 3D mesh, use `pip install "foambryo[viewing]"`.

In this notebook we'll show how to visualize the results, so we choose the latter option.

In a terminal, in the environment this notebook runs in (not from the notebook):

```shell
pip install "foambryo[viewing]"
```

scikit-image, used below to read the TIFF files, comes with it (through delaunay-watershed-3d).

# 1) Prepare an inference mesh

Foambryo actually computes tensions and pressures from a datastructure called a DCEL mesh. Foambryo can create such a mesh from 2 sources:
- an instance segmentation mask, using the Delaunay-Watershed-3D algorithm,
- a multimaterial mesh.

**Data.** The example data used below (segmentation masks and meshes, about 590 MB) are not stored in the
repository. Download them from [this link](https://www.dropbox.com/scl/fo/fanim64m3dwy6in9dkq6v/AGnqONanjcZUMEtvFNFy9wo?rlkey=mx7ffzbea3v93rsqrkdr9ok1q&dl=1) and unzip the archive into
the repository's `data/` folder, so that `data/Meshes/` and `data/Segmentations/` exist; this notebook reads
them from `../data/`.

We provide easy functions to have a mesh ready for force inference:

In [ ]:
import skimage.io as io
from dw3d.io import load_rec

from foambryo import dcel_mesh_from_file, dcel_mesh_from_multimaterial_mesh, dcel_mesh_from_segmentation_mask

# 1) From a segmentation mask
image_path = "../data/Segmentations/Benchmark_labels.tif"
segmentation_mask = io.imread(image_path)  # open segmentation image.
# The data are downloaded separately into data/: see the instructions above.

# The DCEL mesh, ready for force inference:
dcel_mesh_1 = dcel_mesh_from_segmentation_mask(segmentation_mask, min_distance=3, print_info=True)
# Note: mesh reconstruction from a segmentation mask can take some time.

# 2) From a multimaterial mesh file (3D triangle mesh + material data)
dcel_mesh_2 = dcel_mesh_from_file("../data/Meshes/simple_mesh.vtk")

# 3) From an opened multimaterial mesh (3 arrays: points, triangles, labels/materials)
# Here we open a mesh to simulate the situation:
points, triangles, labels = load_rec("../data/Meshes/Benchmark_original_mesh.rec")
dcel_mesh_3 = dcel_mesh_from_multimaterial_mesh(points, triangles, labels)

# 2) Force Inference

Now that we have a DCEL mesh, we can easily compute relative pressures between cells and relative tensions at interfaces between cells (or with the exterior region).

In [ ]:
from foambryo import infer_forces

# select mesh
dcel_mesh = dcel_mesh_1

# infer both tensions and pressures
tensions, pressures = infer_forces(dcel_mesh)

The outputs are simple data structures:

In [ ]:
# pressures is a dictionary that maps cell index to pressure force. 0 is the exterior.
print(f"{pressures=}")

# tensions is a dictionary that maps cells interfaces (cell index 1, cell index 2) to tension force. 0 is the exterior.
print(f"{tensions=}")

You can also compute tensions only, but for the pressures, you need to have first computed the tensions:

In [ ]:
from foambryo import infer_pressures, infer_tensions

tensions = infer_tensions(dcel_mesh)

pressures = infer_pressures(dcel_mesh, tensions)

An important thing to understand is that we compute relative tensions and pressures. As such, when computing pressures, you need to give a base pressure (the pressure of the exterior, the medium) ; when computing tensions, you need to give a mean tension.

By default, the base pressure is 0 while the mean tension is 1. You can always set it as desired:

In [ ]:
# previous lines were equivalent to these:
tensions, pressures = infer_forces(dcel_mesh, mean_tension=1, base_pressure=0)
tensions = infer_tensions(dcel_mesh, mean_tension=1)
pressures = infer_pressures(dcel_mesh, tensions, base_pressure=0)

Finally, foambryo implements several strategies to compute tensions and pressures. The default strategies give good results, but you can explore the other strategies if you want.

In [ ]:
from foambryo import PressureComputationMethod, TensionComputationMethod

print(f"Available pressure computation methods: {[method.name for method in PressureComputationMethod]}.")

print(f"Available tension computation methods: {[method.name for method in TensionComputationMethod]}.")

# previous lines were equivalent to these:
tensions, pressures = infer_forces(
    dcel_mesh,
    mode_tension=TensionComputationMethod.YoungDupre,
    mode_pressure=PressureComputationMethod.Variational,
)
tensions = infer_tensions(dcel_mesh, mode=TensionComputationMethod.YoungDupre)
pressures = infer_pressures(dcel_mesh, tensions, mode=PressureComputationMethod.Variational)

# 3) Plotting forces

Now that we have tensions and pressures, foambryo can also plot the mesh with the values.

To plot pressures:

In [ ]:
from foambryo.viewing import plot_pressure_inference

# Note: only the mesh is necessary. If pressures are not provided, they'll be computed.
plot_pressure_inference(dcel_mesh, pressures)

To plot tensions:

In [ ]:
from foambryo.viewing import plot_tension_inference

# Note: only the mesh is necessary. If tensions are not provided, they'll be computed.
# For tensions, it's better to scatter the cells
plot_tension_inference(dcel_mesh, tensions, scattered=True, scatter_coeff=1)

To plot both on the same viewer:

In [ ]:
from foambryo.viewing import plot_force_inference

plot_force_inference(dcel_mesh, tensions, pressures)

Many more data can be plotted, such as the curvature, the area and volume derivatives...

In [ ]:
plot_force_inference(dcel_mesh, tensions, pressures, scalar_quantities=True)

Plotting functions have many interesting parameters, don't hesitate to play with them !